# Imports

In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score,  StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder

from sklearn.compose import ColumnTransformer

# Classification Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.neighbors import KNeighborsClassifier

# Model Evaluation
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                           f1_score, roc_auc_score, confusion_matrix, 
                           classification_report, roc_curve)

from sklearn.inspection import permutation_importance

import warnings
warnings.filterwarnings('ignore')

# Exploring data

In [25]:
df = pd.read_csv("D:/UNEEQ Interns/Health Care Diagnosis/Data/healthcare_dataset.csv")

In [26]:
#Figuting out data information
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Name                10000 non-null  object 
 1   Age                 10000 non-null  int64  
 2   Gender              10000 non-null  object 
 3   Blood Type          10000 non-null  object 
 4   Medical Condition   10000 non-null  object 
 5   Date of Admission   10000 non-null  object 
 6   Doctor              10000 non-null  object 
 7   Hospital            10000 non-null  object 
 8   Insurance Provider  10000 non-null  object 
 9   Billing Amount      10000 non-null  float64
 10  Room Number         10000 non-null  int64  
 11  Admission Type      10000 non-null  object 
 12  Discharge Date      10000 non-null  object 
 13  Medication          10000 non-null  object 
 14  Test Results        10000 non-null  object 
dtypes: float64(1), int64(2), object(12)
memory usage: 1.1+

In [27]:
#Seeing number of rows and columns in the dataset 
df.shape

(10000, 15)

In [28]:
#Seeing sample of the dataset
df.head()

,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,Tiffany Ramirez,81,Female,O-,Diabetes,2022-11-17,Patrick Parker,Wallace-Hamilton,Medicare,37490.983364,146,Elective,2022-12-01,Aspirin,Inconclusive
1,Ruben Burns,35,Male,O+,Asthma,2023-06-01,Diane Jackson,"Burke, Griffin and Cooper",UnitedHealthcare,47304.064845,404,Emergency,2023-06-15,Lipitor,Normal
2,Chad Byrd,61,Male,B-,Obesity,2019-01-09,Paul Baker,Walton LLC,Medicare,36874.896997,292,Emergency,2019-02-08,Lipitor,Normal
3,Antonio Frederick,49,Male,B-,Asthma,2020-05-02,Brian Chandler,Garcia Ltd,Medicare,23303.322092,480,Urgent,2020-05-03,Penicillin,Abnormal
4,Mrs. Brandy Flowers,51,Male,O-,Arthritis,2021-07-09,Dustin Griffin,"Jones, Brown and Murray",UnitedHealthcare,18086.344184,477,Urgent,2021-08-02,Paracetamol,Normal


In [29]:
#Checking daatset statistics
df.describe()

,Age,Billing Amount,Room Number
count,10000.000000,10000.000000,10000.000000
mean,51.452200,25516.806778,300.082000
std,19.588974,14067.292709,115.806027
min,18.000000,1000.180837,101.000000
25%,35.000000,13506.523967,199.000000
50%,52.000000,25258.112566,299.000000
75%,68.000000,37733.913727,400.000000
max,85.000000,49995.902283,500.000000


In [30]:
#Checking for null values in the dataset
df.isnull().sum()

Name                  0
Age                   0
Gender                0
Blood Type            0
Medical Condition     0
Date of Admission     0
Doctor                0
Hospital              0
Insurance Provider    0
Billing Amount        0
Room Number           0
Admission Type        0
Discharge Date        0
Medication            0
Test Results          0
dtype: int64

# Data Feature Engineering

In [31]:
#converting date columns 
date_columns = ['Date of Admission', 'Discharge Date']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

#Chaning age to numeric values
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')

#creating age groups 
df['Age group'] = pd.cut(df['Age'], bins=[0, 18, 35, 50, 65,], labels=['0-18', '19-35', '36-50', '51-65'])#

#Assigning target variable
target_col = 'Test Results'

#Selecting feature columns and target variable
feature_columns = ['Age' , 'Gender' , 'Blood Type' , 'Medical Condition' , 'Admission Type' , 'Medication']

#Separating features and target variable
x = df[feature_columns].copy()
y =  df[target_col]



In [32]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

In [33]:
min_class_size = y_train.value_counts().min()
n_classes = y_train.nunique()
print(f"\nNumber of classes: {n_classes}")
print(f"Smallest class size: {min_class_size}")


Number of classes: 3
Smallest class size: 2614


# Model training

In [ ]:
# Preprocessing the data
numerical_features = ['Age']
categorical_features = ['Gender', 'Blood Type', 'Medical Condition', 'Admission Type', 'Medication']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

# Transforming the data
x_train_processed = preprocessor.fit_transform(x_train)
x_test_processed = preprocessor.transform(x_test)

# Defining all potential models 
models = {
    'Logistic Regression' : LogisticRegression(max_iter= 1000 ,random_state=42 , class_weight='balanced'),
    'RandomForestClassifier' : RandomForestClassifier(n_estimators=100 ,random_state=42 , class_weight='balanced'),
    'GradientBoostingClassifier' : GradientBoostingClassifier(n_estimators=100 ,random_state=42),
    'KNN' : KNeighborsClassifier(n_neighbors=5)
}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

# Evaluating each model using cross-validation and validation set   
for name , modle in models.items():
    print(f"Evaluating {name}...")
    
    # Cross-validation on training data
    cv_scores = cross_val_score(modle, x_train_processed, y_train, cv=cv_strategy, scoring='accuracy' , n_jobs=-1)
    modle.fit(x_train_processed, y_train)
    Y_val_pred = modle.predict(x_test_processed)

    # Storing results
    results[name] = {
            'CV Accuracy Mean': cv_scores.mean(),
            'CV Accuracy Std': cv_scores.std(),
            'Validation Accuracy': accuracy_score(y_test, Y_val_pred),
            'Validation F1 (Macro)': f1_score(y_test, Y_val_pred, average='macro', zero_division=0),
            'Validation F1 (Weighted)': f1_score(y_test, Y_val_pred, average='weighted', zero_division=0)
          }
        
    print(f"  ✅ Validation Accuracy: {results[name]['Validation Accuracy']:.4f}")
    print(f"  ✅ Validation F1 (Macro): {results[name]['Validation F1 (Macro)']:.4f}")


# Display results
results_df = pd.DataFrame(results).T
print("\n" + "="*80)
print("MODEL PERFORMANCE COMPARISON FOR MEDICAL CONDITION PREDICTION")
print("="*80)
print(results_df.round(4))


 # Identify best model
best_model_name = results_df['Validation Accuracy'].idxmax()
best_model = models[best_model_name]

print(f"\n🏆 Best model: {best_model_name}")
print(f"   Validation Accuracy: {results_df.loc[best_model_name, 'Validation Accuracy']:.4f}")
print(f"   Validation F1 (Macro): {results_df.loc[best_model_name, 'Validation F1 (Macro)']:.4f}")

Evaluating Logistic Regression...
  ✅ Validation Accuracy: 0.3265
  ✅ Validation F1 (Macro): 0.3265
Evaluating RandomForestClassifier...
  ✅ Validation Accuracy: 0.3405
  ✅ Validation F1 (Macro): 0.3397
Evaluating GradientBoostingClassifier...
  ✅ Validation Accuracy: 0.3535
  ✅ Validation F1 (Macro): 0.3457
Evaluating KNN...
  ✅ Validation Accuracy: 0.3285
  ✅ Validation F1 (Macro): 0.3162

MODEL PERFORMANCE COMPARISON FOR MEDICAL CONDITION PREDICTION
                            CV Accuracy Mean  CV Accuracy Std  \
Logistic Regression                   0.3240           0.0047   
RandomForestClassifier                0.3356           0.0089   
GradientBoostingClassifier            0.3368           0.0095   
KNN                                   0.3431           0.0084   

                            Validation Accuracy  Validation F1 (Macro)  \
Logistic Regression                      0.3265                 0.3265   
RandomForestClassifier                   0.3405                 0.339

In [35]:
y_test_pred = best_model.predict(x_test_processed)

# Create dataframe with features and predictions
results_df = x_test.copy()
results_df['Predicted_Test_Results'] = y_test_pred

# Display predictions with features
print("\nPredictions with Features (first 10 rows):")
print(results_df.head(10))



Predictions with Features (first 10 rows):
      Age  Gender Blood Type Medical Condition Admission Type   Medication  \
7301   59  Female        AB+          Diabetes         Urgent    Ibuprofen   
490    38  Female         O+            Asthma      Emergency  Paracetamol   
4738   78    Male         B-            Cancer       Elective    Ibuprofen   
9997   54    Male         B-         Arthritis       Elective    Ibuprofen   
6054   31  Female         A-         Arthritis       Elective      Aspirin   
8846   79    Male         A+         Arthritis       Elective    Ibuprofen   
7770   23  Female         O+            Cancer      Emergency      Aspirin   
9192   70  Female         B+          Diabetes      Emergency      Lipitor   
3332   42    Male         O-         Arthritis         Urgent    Ibuprofen   
2613   59    Male         A-            Asthma       Elective  Paracetamol   

     Predicted_Test_Results  
7301               Abnormal  
490                Abnormal  
4738   